Fron N-Grams to logistic regresssion 
### Dataset Setup

The IMDB dataset contains movie reviews labeled as positive or negative. we used a balanced subset of 5,000 training reviews and 1,000 test reviews. The training and test sets are sampled separately, and a fixed random seed is used to make the selection reproducible.

In [1]:
import re
import html
import numpy as np
from collections import Counter, defaultdict

from datasets import load_dataset

imdb = load_dataset("stanfordnlp/imdb")

print(imdb)
print(imdb["train"][0]["text"][:300])

c:\Users\jana\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})
I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really h


In [2]:
#from datasets import load_dataset

#imdb = load_dataset("imdb")

#print(imdb)
#print(imdb["train"][0])

inspecting the raw data

In [3]:
print("Training reviews:", len(imdb["train"]))
print("Test reviews:", len(imdb["test"]))

for i in range(5):
    review = imdb["train"][i]
    print("\nLabel:", review["label"])
    print("Review:", review["text"][:500])
    from collections import Counter



Training reviews: 25000
Test reviews: 25000

Label: 0
Review: I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attent

Label: 0
Review: "I Am Curious: Yellow" is a risible and pretentious steaming pile. It doesn't matter what one's political views are because this film can hardly be taken seriously on any level. As for the claim that frontal male nudity is an automatic NC-17, that isn't true. I've seen R-rated films with male nudity. Granted, they only offer some fleeting views, but where are the R-rated films with gaping vulvas and flapping labia? 

In [4]:
#Extract the exact balanced subset
def extract_balanced_subset(dataset, samples_per_class, seed=42):
    texts = []
    labels = []

    for label in [0, 1]:
        class_indices = [
            i for i, item in enumerate(dataset)
            if item["label"] == label
        ]

        rng = np.random.default_rng(seed + label)
        chosen_indices = rng.choice(
            class_indices,
            size=samples_per_class,
            replace=False
        )

        for index in chosen_indices:
            texts.append(dataset[int(index)]["text"])
            labels.append(label)

    # Shuffle the combined subset
    rng = np.random.default_rng(seed)
    order = rng.permutation(len(texts))

    texts = [texts[i] for i in order]
    labels = np.array([labels[i] for i in order])

    return texts, labels


train_texts, y_train = extract_balanced_subset(
    imdb["train"], 2500, seed=42
)

test_texts, y_test = extract_balanced_subset(
    imdb["test"], 500, seed=123
)

print("Training:", len(train_texts))
print("Training negative:", np.sum(y_train == 0))
print("Training positive:", np.sum(y_train == 1))

print("Testing:", len(test_texts))
print("Testing negative:", np.sum(y_test == 0))
print("Testing positive:", np.sum(y_test == 1))

Training: 5000
Training negative: 2500
Training positive: 2500
Testing: 1000
Testing negative: 500
Testing positive: 500


### Text Preprocessing and Tokenization

Text preprocessing prepares the reviews for language modeling. I decode HTML entities, remove HTML tags, convert text to lowercase, and tokenize the reviews into words using regular expressions. I preserve negation words such as "not" because they can change the sentiment of a review. I also add `<s>` and `</s>` tokens to mark the beginning and end of each review.

In [5]:
#preprocessing and tokenization
def clean_text(text):
    # Decode HTML entities
    text = html.unescape(text)

    # Remove HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # Normalize case
    text = text.lower()

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [6]:
def tokenize(text):
    text = clean_text(text)

    # Keep alphabetic words and internal apostrophes
    tokens = re.findall(r"[a-z]+(?:'[a-z]+)?", text)

    # Add sentence boundary markers
    return ["<s>"] + tokens + ["</s>"]

In [7]:
#testing
example = "<p>This MOVIE isn't bad! I loved it.</p>"

print(clean_text(example))
print(tokenize(example))

this movie isn't bad! i loved it.
['<s>', 'this', 'movie', "isn't", 'bad', 'i', 'loved', 'it', '</s>']


In [10]:
#: Keeping the original text and labels for evaluation
# but  the tokenized versions for all subsequent language-model calculations.

X_train_tokens = [tokenize(text) for text in train_texts]
X_test_tokens = [tokenize(text) for text in test_texts]

print("Original review:")
print(train_texts[0][:300])

print("\nTokenized review:")
print(X_train_tokens[0][:40])

print("\nNumber of training reviews:", len(X_train_tokens))
print("Number of test reviews:", len(X_test_tokens))

Original review:
I did not enjoy the film, Joshua, at all. Perhaps it is because I saw another, much better similar film titled Orphan 2 days prior but perhaps it's really just because this film was not very good. I am going with the ladder. Sure, the plot of an evil child is not exactly original but that doesn't me

Tokenized review:
['<s>', 'i', 'did', 'not', 'enjoy', 'the', 'film', 'joshua', 'at', 'all', 'perhaps', 'it', 'is', 'because', 'i', 'saw', 'another', 'much', 'better', 'similar', 'film', 'titled', 'orphan', 'days', 'prior', 'but', 'perhaps', "it's", 'really', 'just', 'because', 'this', 'film', 'was', 'not', 'very', 'good', 'i', 'am', 'going']

Number of training reviews: 5000
Number of test reviews: 1000
